In [ ]:
from pathlib import Path
import os, sys
for project_root in (Path.cwd(), *Path.cwd().parents):
    if (project_root / 'regarch').is_dir():
        os.chdir(project_root)
        sys.path.insert(0, str(project_root))
        break
else:
    raise RuntimeError('Run this notebook from inside the RegArch project.')

import numpy as np
import pandas as pd

from regarch import FateMap, GRNInference, construct_fate_map, load_fate_map_topology, parse_edge_dict
from regarch.downstream_analysis import (
    cluster_regulatory_interactions, get_high_expression_genes,
    get_regulators_for_target_gene, get_targets_for_regulator_gene,
    identify_differentiation_regulators, identify_fate_bias_regulators,
    summarize_interaction_cluster_specificity,
)
from regarch.plotting import (
    plot_key_genes_differentiation, plot_key_genes_fate_bias,
    plot_regulatory_genes_along_fate_map,
    plot_regulatory_interactions_along_fate_map,
    plot_regulatory_interactions_by_cell_type,
    plot_regulatory_network_along_fate_map,
    plot_regulatory_strength_along_fate_map, plot_target_genes_along_fate_map,
    rank_regulators_by_target_count,
)
from regarch.utils.data_io import get_dynamic_networks

# Cached, fate-map-consistent GRNs make the default demo fast.
RECOMPUTE_FATE_MAP = False
RECOMPUTE_GRNS = False

### Reconstructing cell fate map

In [ ]:
barcode_path = 'examples/data/zebrafish/input_data/barcode_site.csv'
cell_types_path = 'examples/data/zebrafish/input_data/cell_types.csv'
fate_map_path = 'examples/data/zebrafish/additional_input_data/fate_map_topology.csv'
t_S = 15

if RECOMPUTE_FATE_MAP:
    sc_mat = np.asarray(pd.read_csv(barcode_path, index_col=0))
    cell_types = pd.read_csv(cell_types_path, index_col=0).to_dict()['celltypes']
    edges, fate_map = construct_fate_map(
        sc_mat, cell_types, t_S, beta=1, max_iter=5, tol=1e-1,
        output_edges=True,
    )
else:
    edges = parse_edge_dict(load_fate_map_topology(fate_map_path))
    fate_map = FateMap(edges)

### Inferring dynamic gene regulatory networks

In [ ]:
saved_dir = 'examples/results/zebrafish/inferred_grns'
atac_file_path = "examples/data/zebrafish/input_data/atac_data.csv"
expression_file_path = "examples/data/zebrafish/input_data/expression_data.csv"

regulator_names=list(pd.read_csv("examples/data/zebrafish/additional_input_data/regulatory_genes_name.csv")['x'])
target_gene_names=list(pd.read_csv("examples/data/zebrafish/additional_input_data/target_genes_name.csv")['x'])

if RECOMPUTE_GRNS:
    grn_inference_result = GRNInference(atac_file_path, expression_file_path, fate_map, saved_dir)
    grn_inference_result.infer_grn(20)
    target_networks_dict = grn_inference_result.get_target_networks(0)

dynamic_networks_dict = get_dynamic_networks(saved_dir, fate_map, 0.02, regulator_names, target_gene_names)

### Revealing the reconfiguration pattern of inferred GRNs along cell lineages

In [ ]:
regulator_dict = get_regulators_for_target_gene('admb', 'Tcells', saved_dir, regulator_names)
print(regulator_dict)

In [ ]:
target_dict = get_targets_for_regulator_gene('tbx21', 'Tcells', saved_dir, regulator_names)
print(target_dict)

In [ ]:
output_path = 'examples/results/zebrafish/figures/'
path=list(fate_map.nodes.keys())

In [ ]:
plot_regulatory_interactions_along_fate_map(dynamic_networks_dict, list(fate_map.nodes.keys()), output_path, figsize=(1.4,1))

In [ ]:
plot_regulatory_genes_along_fate_map(saved_dir, regulator_names, 'tbx21', list(fate_map.nodes.keys()), 0, output_path, figsize=(1.4,1))

In [ ]:
ordered_genes = rank_regulators_by_target_count(target_gene_names, regulator_names, 'total', saved_dir, path, 0.1)
plot_target_genes_along_fate_map(ordered_genes, regulator_names, 'negative', saved_dir, path, 0.1, output_path)

In [ ]:
ordered_genes

In [ ]:
plot_target_genes_along_fate_map(ordered_genes, regulator_names, 'positive', saved_dir, path, 0.1, output_path)

In [ ]:
plot_target_genes_along_fate_map(ordered_genes, regulator_names, 'total',saved_dir, path, 0.1, output_path)

In [ ]:
path = fate_map.get_path("Endocardium")
plot_regulatory_network_along_fate_map('klf2a', dynamic_networks_dict, path, output_path, 0.12)

In [ ]:
path = fate_map.get_path("Endocardium")
plot_regulatory_strength_along_fate_map(saved_dir, path, regulator_names, 'junbb', output_path)

### Identifying key regulatory genes

#### Identifying regulatory genes key for differentiation

In [ ]:
Endocardium_gene_expression_matrix = pd.read_csv('examples/data/zebrafish/additional_input_data/Exp_Endocardium.csv', index_col=0)
Epicardium_gene_expression_matrix = pd.read_csv('examples/data/zebrafish/additional_input_data/Exp_Epicardium.csv', index_col=0)
Fibroblasts_gene_expression_matrix = pd.read_csv('examples/data/zebrafish/additional_input_data/Exp_Fibroblasts.csv', index_col=0)
Smoothmusclecells_gene_expression_matrix = pd.read_csv('examples/data/zebrafish/additional_input_data/Exp_muscle.csv', index_col=0)
Macrophages_gene_expression_matrix = pd.read_csv('examples/data/zebrafish/additional_input_data/Exp_Macrophages.csv', index_col=0)
Tcells_gene_expression_matrix = pd.read_csv('examples/data/zebrafish/additional_input_data/Exp_tcell.csv', index_col=0)

Endocardium_gene_expression_matrix.index = Epicardium_gene_expression_matrix.index = Fibroblasts_gene_expression_matrix.index = Smoothmusclecells_gene_expression_matrix.index = Macrophages_gene_expression_matrix.index = Tcells_gene_expression_matrix.index = target_gene_names + regulator_names

In [ ]:
Endocardium_genes = [i for i in get_high_expression_genes(Endocardium_gene_expression_matrix) if i in(target_gene_names)]
Epicardium_genes = [i for i in get_high_expression_genes(Epicardium_gene_expression_matrix) if i in(target_gene_names)]
Fibroblasts_genes = [i for i in get_high_expression_genes(Fibroblasts_gene_expression_matrix) if i in(target_gene_names)]
Smoothmusclecells_genes = [i for i in get_high_expression_genes(Smoothmusclecells_gene_expression_matrix) if i in(target_gene_names)]
Macrophages_genes = [i for i in get_high_expression_genes(Macrophages_gene_expression_matrix) if i in(target_gene_names)]
Tcells_genes = [i for i in get_high_expression_genes(Tcells_gene_expression_matrix) if i in(target_gene_names)]

In [ ]:
high_expression_gene_dict = {'Endocardium':Endocardium_genes, 'Epicardium':Epicardium_genes, 'Fibroblasts':Fibroblasts_genes, 'Smoothmusclecells':Smoothmusclecells_genes, 'Macrophages':Macrophages_genes, 'Tcells':Tcells_genes, 'IP1':None, 'IP2':None, 'IP3':None, 'IP4':None, 'IP5':None}
fate_map = FateMap(edges, high_expression_gene_dict)

In [ ]:
key_regulators_df_concat = pd.DataFrame()

nodes = fate_map.node_internals[::-1]+[fate_map.node_root]
for node in nodes:
    key_regulators = identify_differentiation_regulators(saved_dir, fate_map, 0.11, regulator_names, target_gene_names, node)
    high_expression_genes_of_ancestor_node = list(set(key_regulators.loc[((key_regulators['DRS']>0.35) & (key_regulators['DRC']>0.6)),'regulator_id']) & set(target_gene_names))
    high_expression_gene_dict[node] = high_expression_genes_of_ancestor_node
    fate_map = FateMap(edges,high_expression_gene_dict)
    key_regulators_df_concat = pd.concat([key_regulators_df_concat, key_regulators])

In [ ]:
nodes = list(key_regulators_df_concat['node_id'].drop_duplicates())
key_regulators_df_concat_V2 = key_regulators_df_concat.fillna(0)
plot_key_genes_differentiation(key_regulators_df_concat_V2, nodes, regulator_names, output_path)

#### Identifying regulatory genes key for fate bias

In [ ]:
ancestor_node_id = 'IP3'
child_nodes = [fate_map.nodes[ancestor_node_id].directed_edges[i].end for i in range(2)]
print(child_nodes)
high_expression_target_genes_in_child_1 = fate_map.nodes[child_nodes[0]].high_expression_genes_in_leaf
high_expression_target_genes_in_child_2 = fate_map.nodes[child_nodes[1]].high_expression_genes_in_leaf
 
target_genes_1 = list(set(target_gene_names) & set(high_expression_target_genes_in_child_1) - set(high_expression_target_genes_in_child_2))
target_genes_2 = list(set(target_gene_names) & set(high_expression_target_genes_in_child_2) - set(high_expression_target_genes_in_child_1))


key_regulators_df_IP3 = identify_fate_bias_regulators(dynamic_networks_dict, ancestor_node_id, target_genes_1, target_genes_2, regulator_names)

In [ ]:
df1 = key_regulators_df_IP3[0][['Regulatory gene', 'FBI', 'FBP']].reset_index(drop=True)
df2 = key_regulators_df_IP3[1][['Regulatory gene', 'FBI', 'FBP']]
merged_df = df1.merge(
    df2, 
    on = 'Regulatory gene', 
    how = 'left',  
    suffixes = (f'_{child_nodes[0]}', f'_{child_nodes[1]}')
)

plot_key_genes_fate_bias(merged_df, child_nodes, output_path, figsize=(1.5, 3))

### Identifying constitutive and specific regulatory interactions

In [ ]:
X, centers, weight_matrix = cluster_regulatory_interactions(saved_dir, fate_map, 0.1, regulator_names, target_gene_names, len(regulator_names), len(target_gene_names), 4, 2)

In [ ]:
edges_cluster_to_nodes = summarize_interaction_cluster_specificity(saved_dir, fate_map, 0.1, regulator_names, target_gene_names, 4, 0.4, X, len(regulator_names), len(target_gene_names), weight_matrix)

In [ ]:
plot_regulatory_interactions_by_cell_type(edges_cluster_to_nodes, output_path)